In [2]:
import cv2
import numpy as np
import torch as t
import torchvision
import torchaudio
import albumentations as A
from ultralytics import YOLO

In [21]:
video_path = '/home/var-roman/Desktop/my_projects/diploma_project/data/videos/Japan_vs_Poland_ultrashort.mp4'

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Помилка: Не вдалося відкрити відео!")
    exit()

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Загальна кількість кадрів у відео: {total_frames}")

TARGET_FRAME = 1111

if TARGET_FRAME >= total_frames:
    cap.release()
    raise ValueError(f"Зупинка! Ти просиш кадр {TARGET_FRAME}, а у відео їх всього {total_frames}.")

cap.set(cv2.CAP_PROP_POS_FRAMES, TARGET_FRAME)
ret, frame = cap.read()
cap.release()

if not ret:
    print("Помилка: Не вдалося прочитати кадр з відео!")
    exit()

cv2.namedWindow("Calibrator", cv2.WINDOW_NORMAL)
cv2.imshow("Calibrator", frame)

while True:
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cv2.destroyAllWindows()

Загальна кількість кадрів у відео: 1135


In [4]:
model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/training_models/models/main_model_april.pt')

In [5]:
results = model.predict(
    source=frame,
    imgsz=1920,
    conf=0.3,
    iou=0.45,
    show=True
)


0: 1088x1920 1 volleyball, 80.5ms
Speed: 5.6ms preprocess, 80.5ms inference, 10.3ms postprocess per image at shape (1, 3, 1088, 1920)


In [6]:
pts_real_3d = np.array([
    [0.0, 0.0, 0.0],
    [9.0, 0.0, 0.0],
    [9.0, 18.0, 0.0],
    [0.0, 18.0, 0.0]],dtype=np.float32)

pts_video_2d = np.array([
    [73, 1031],
    [1835, 1027],
    [1504, 606],
    [405, 606]],dtype=np.float32)

K = np.array([
    [1300.0, 0.0,    960.0],
    [0.0,    1300.0, 540.0],
    [0.0,    0.0,    1.0]],dtype=np.float32)

dist_coeffs = np.zeros((4, 1))

def calibrate_camera(pts_3d, pts_2d, camera_matrix, dist):
    success, rvec, tvec = cv2.solvePnP(pts_3d, pts_2d, camera_matrix, dist, flags=cv2.SOLVEPNP_ITERATIVE)

    if not success:
        raise ValueError("solvePnP не зміг знайти рішення! Перевір порядок точок.")

    R, _ = cv2.Rodrigues(rvec)

    # Знаходимо фізичну позицію камери в залі (C = -R^T * tvec)
    R_inv = np.linalg.inv(R)
    camera_position = -np.dot(R_inv, tvec)

    return R, tvec, camera_position

# Виконуємо калібрування один раз для всього відео
R_matrix, t_vec, camera_pos = calibrate_camera(pts_real_3d, pts_video_2d, K, dist_coeffs)

print("Матриця обертання R:\n", R_matrix)
print("\nПозиція камери в залі (X, Y, Z) в метрах:\n", camera_pos.ravel())

Матриця обертання R:
 [[    0.99998  -0.0054972  0.00028705]
 [-0.00010467   -0.071125    -0.99747]
 [  0.0055037     0.99745   -0.071125]]

Позиція камери в залі (X, Y, Z) в метрах:
 [     4.4765     -6.2931      2.8991]


In [18]:
def get_3d_position(u, v, bbox_w, K, R_matrix, camera_pos, ball_diameter=0.21):
    """
    Параметри:
    u, v: центр BBox м'яча в пікселях
    bbox_w: ширина BBox м'яча в пікселях (беремо ширину, бо вона менше страждає від Motion Blur, ніж висота)
    K: матриця камери (Intrinsic)
    R_matrix: матриця обертання з solvePnP
    camera_pos: 3D позиція камери (C) з solvePnP
    ball_diameter: діаметр волейбольного м'яча в метрах (стандарт 21 см)
    """
    f_x = K[0, 0]
    c_x = K[0, 2]
    f_y = K[1, 1]
    c_y = K[1, 2]

    Z_c = (f_x * ball_diameter) / bbox_w

    x_norm = (u - c_x) / f_x
    y_norm = (v - c_y) / f_y
    d_cam = np.array([[x_norm], [y_norm], [1.0]])

    d_world = np.dot(R_matrix.T, d_cam)
    P_world = camera_pos.reshape(3, 1) + Z_c * d_world

    return P_world.ravel()

In [19]:
u_cam, v_cam, w_cam = np.ravel(np.asarray(results[0].boxes.xywh.cpu(), dtype=np.float32))[:3]

get_3d_position(u_cam, v_cam, w_cam, K, R_matrix, camera_pos, ball_diameter=0.21)

array([     3.1129,     0.78706,      4.2089])

In [26]:
print(results)

[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'volleyball'}
obb: None
orig_img: array([[[  8,   5,   6],
        [  8,   5,   6],
        [  8,   5,   6],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[  8,   5,   6],
        [  8,   5,   6],
        [  8,   5,   6],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       [[  6,   3,   4],
        [  6,   3,   4],
        [  6,   3,   4],
        ...,
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]],

       ...,

       [[ 33,  33,  92],
        [ 34,  34,  93],
        [ 37,  35, 103],
        ...,
        [113,  51,  13],
        [112,  50,  12],
        [112,  50,  12]],

       [[ 34,  34,  93],
        [ 34,  34,  93],
        [ 34,  32, 100],
        ...,
        [113,  51,  13],
        [113,  51,  13],
        